<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Vegetation Times series extraction class - Dev notebook

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")


## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name','geometry']].head())

In [ ]:
# Export to CSV
output_file = os.path.join(manager.output_result_dir, "ndvi_specific_dates_export.csv")
manager.sfd_list.to_csv(output_file, index=False)
print(f"✅ Exported to: {output_file}")

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from VTS_service_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.extractors.VTS_functions import VegationTsExtractor
extractor = VegationTsExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id", "start_date": "sowingDate"}

extractor.setup_vegetation_ts_parameters(
        start_date= "2021-01-01",
        end_date="2026-01-01",
        vegetation_index= "NDVI",
        is_extrapolated=True,
        limit=3000,
        extraction_mode="windows",        # Options: period, specific_dates or windows
        target_dates=[], 
        historical_years=10,
        partial_frequency=50,
        column_mapping=column_mapping,
        # kpi_filter={
        #         'kpi_name': 'Summer NDVI Accumulation',
        #         'aggregation': 'accumulation'
        #         }
        )
extractor2 = VegationTsExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )
extractor2.setup_vegetation_ts_parameters(
        start_date= "2021-01-01",
        end_date="2026-01-01",
        vegetation_index= "NDVI",
        is_extrapolated=True,
        limit=3000,
        extraction_mode="windows",        # Options: period, specific_dates or windows
        target_dates=["2021-01-01","2026-01-01"], 
        historical_years=10,
        partial_frequency=50,
        column_mapping=column_mapping,
        kpi_filter={}
        )

### 🗺️ Test extraction for specific dates

In [ ]:
# Run specific dates extraction
result = extractor.process_specific_dates_bulk_parallel(
    entity_list=manager.sfd_list,  # Uses id and name columns
    params=None,
    max_workers=30,
    output_path=manager.output_result_dir,
    fail_safe=False,
    prefix="ndvi_specific_dates"
)

print(f"\nResults summary:")
print(f"Total entities: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")
print(f"Target dates: {len(result['target_dates'])} dates")

# Display results (format: date, field_name, ndvi_value)
results_df = result["results_df"]
print(f"\n📊 Output:")
print(results_df.head(20))

### 🗺️ Test export result for specific dates

In [ ]:
results_df = result["results_df"]
print(f"Total rows: {len(results_df)}")
print(f"Columns: {list(results_df.columns)}")

# Export to CSV
output_file = os.path.join(manager.output_result_dir, "ndvi_specific_dates_export.csv")
results_df.to_csv(output_file, index=False)
print(f"✅ Exported to: {output_file}")

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33", #for VTS please make sure you are using field id available for your user
    "geometry": "POLYGON ((-57.17400567 -33.70070656, -57.17404544 -33.70089983, -57.17426497 -33.70085284, -57.1751335 -33.700639620000004, -57.17621637 -33.70034084, -57.176741660000005 -33.70013531, -57.17750935 -33.69988044, -57.17758905 -33.69976989, -57.179569210000004 -33.69782629, -57.17962191 -33.69772377, -57.17902293 -33.6971412, -57.17789108 -33.69631739, -57.17602474 -33.69811988, -57.175453260000005 -33.69861127, -57.17495038 -33.69896085, -57.17434281 -33.69944557, -57.17369835 -33.6998999, -57.17379511 -33.70024369, -57.17389311 -33.70048781, -57.17400567 -33.70070656))",
    "crop.id": "SOYBEANS",
    # "start_date":"2025-06-01",
    # "end_date":"2025-10-01",
    'years': [2024, 2023, 2022]
}


#### Test get_vegetation_ts_api

In [ ]:
print("\n--- get vegetation time series - mode period ---")
try:
    result = extractor.get_vegetation_api(
        entity_data=seasonfield_data
    )
    
    print("✅ Vegetation time series data retrieved successfully!")
    if isinstance(result, dict) and 'value' in result:
        records = result['value']
        print(f"Number of records: {len(records)}")
        if records:
            print(f"\n📊 First record:")
            for key, value in list(records[0].items())[:5]:
                print(f"  {key}: {value}")
    else:
        print(f"Response: {result}")
    
except Exception as e:
    print(f"❌ API call failed: {e}")
    print(f"📄 API Error Response: {e.response.text}")  
    import traceback
    traceback.print_exc()

In [ ]:
print("\n--- get vegetation time series - mode Windows ---")
try:
    result = extractor2.get_vegetation_api(
        entity_data=seasonfield_data
    )
    
    print("✅ Vegetation time series data retrieved successfully!")
    if isinstance(result, dict) and 'value' in result:
        records = result['value']
        print(f"Number of records: {len(records)}")
        if records:
            print(f"\n📊 First record:")
            for key, value in list(records[0].items())[:5]:
                print(f"  {key}: {value}")
    else:
        print(f"Response: {result}")
    
except Exception as e:
    print(f"❌ API call failed: {e}")
    print(f"📄 API Error Response: {e.response.text}")  
    import traceback
    traceback.print_exc()

#### Test get_vegetation_ts_api_safe

In [ ]:
print("\n--- Test: get_vegation_ts_safe ---")
safe_result = extractor.get_vegetation_api_safe(seasonfield_data)
print(safe_result)

In [ ]:
print("\n--- Test: get_vegation_ts_safe ---")
safe_result2 = extractor2.get_vegetation_api_safe(seasonfield_data)
print(safe_result)

#### Test format_vegetation_ts_json

In [ ]:
print("\n--- Test: format_vegetation_ts_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_vegetation_ts_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head(500))
else:
    print("⚠️ Skipping format_vegetation_ts_json: No valid data from API.")

In [ ]:
print("\n--- Test: format_vegetation_ts_json ---")
if safe_result2["success"] and safe_result["data"]:
    formatted_df2 = extractor.format_vegetation_ts_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df2.head(500))
else:
    print("⚠️ Skipping format_vegetation_ts_json: No valid data from API.")

#### Test filter_timeseries_kpi

In [ ]:
# Add this import at the top of your notebook
from earthdaily.agriculture.core.api_utils import filter_timeseries_kpi, format_kpi_results

print("\n--- Test: filter_timeseries_kpi ---")

if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_vegetation_ts_json(safe_result["data"])
    
    if not formatted_df.empty:
        try:
            # Call the function directly (not as a method)
            kpi_result = filter_timeseries_kpi(
                timeseries_df=formatted_df,
                start_date='2025-06-01',
                end_date='2025-08-31',
                kpi_name='NDVI Accumulation',
                aggregation='accumulation',
                years=[2024, 2023, 2022]
            )
            
            print("✅ KPI computation successful!")
            print(f"\n📊 KPI Summary:")
            print(f"  Current Value: {kpi_result['current_period']['value']}")
            print(f"  Historical Avg: {kpi_result['historical_avg']['value']}")
            print(f"  Difference: {kpi_result['comparison']['difference']}")
            print(f"  Change %: {kpi_result['comparison']['percent_change']}%")
            
            # Format results
            kpi_df = format_kpi_results(kpi_result)
            print(f"\n📋 Formatted KPI Results:")
            display(kpi_df)
            
        except Exception as e:
            print(f"❌ KPI computation failed: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️ Formatted DataFrame is empty")
else:
    print("⚠️ Skipping filter_timeseries_kpi: No valid data from API.")

### 🗺️ process_single_entity

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "3a5yn53",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "SOYBEANS",
    # "start_date":"2025-06-01",
    # "end_date":"2025-10-01",
    # 'years': [2024, 2023, 2022]
}).to_dict() 

result = extractor.process_single_entity_vegetation_ts(row)

print(result)

In [ ]:
result = extractor.process_single_entity_vegetation_ts(manager.sfd_list.iloc[0])
print(result)

In [ ]:
result = extractor2.process_single_entity_vegetation_ts(row)
print(result)

### Test historical years validation (list, string, column_mapping)

Validates that `years` works as a native list, comma-separated string (pipeline flattening),
and via `column_mapping` remapping from `historical_seasons`.

In [ ]:
from earthdaily.agriculture.core.api_utils import validate_historical_years

# Test validate_historical_years directly
test_cases = [
    None,
    'ALL',
    [2024, 2023, 2022],
    '2024,2023,2022',
    5,
]

for tc in test_cases:
    result = validate_historical_years(tc)
    print(f'  {str(tc):30s} -> {result} ({type(result).__name__})')


### 🗺️ process_vegetation_TS_bulk_extraction_parallel

In [ ]:
from datetime import timedelta
import pandas as pd
top25 = manager.sfd_list.head(50)

# # Rename column
# top25 = top25.rename(columns={'sowingDate': 'start_date'})
# top25=top25.rename(columns={"crop.id": "crop"})

# # Convert start_date to datetime if not already
# top25['start_date'] = pd.to_datetime(top25['start_date'])

# # Add end_date as start_date + 50 days
# top25['end_date'] = top25['start_date'] + pd.Timedelta(days=50)

# # Convert back to string format (YYYY-MM-DD) if needed for your API
# top25['start_date'] = top25['start_date'].dt.strftime('%Y-%m-%d')
# top25['end_date'] = top25['end_date'].dt.strftime('%Y-%m-%d')
# print(top25.columns)
# Launch extraction with 10 threads 

result = extractor.process_entity_vegetation_ts_bulk_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    # filter_column="crop",
    # filter_value="CORN",
    # filter_type="exclude" # filter type used to 'include' or 'exclude' row matching column and value filter
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)

In [ ]:
print(results.head)
